In [1]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

D:\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [3]:
DATASET_SAVED = "raw_official"
URL_INDICATORS = "urls"

In [5]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [6]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [7]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'W3_pc_front_C', 'E14b_movies_anti_tob_ads', 'R_admin_covert_marking', 'O_Ins_off_hlth_prof', 'NCD_CCS_TobPlan', 'W21_emissions_A', 'P8_compliance', 'E17b_csr_promo_self', 'R_Price_lowest_cost_estimate', 'P6_compliance', 'R_Ad_val_estimate', 'W21_emissions_quant_B', 'Yth_daily_tob_use', 'TOBACCO_MPOWER_R_TAXSTRUCTURE', 'E22_subnational_authority_exists', 'R_Price_mp_estimate_usd', 'W15_specific_hw_B', 'Rev_import', 'R_afford_price_dispersion', 'W11_outside_pack_A', 'R_earmarking', 'W22_emissions_qual_B', 'R_Curr_premium_cost', 'TOBACCO_MPOWER_M3_TOBACCOUSESMOKINGSCHOOL', 'TOBACCO_MPOWER_O1_TREATMENTSUPPORT', 'W-MM_Group', 'O_Ins_hosp', 'TOBACCO_MPOWER_R_TAXREVENUEPREVIOUS', 'P3_compliance', 'R_contains_nicotine', 'Yth_curr_smokeless', 'E18_csr_anti_tobacco_media', 'W5_top_side_B', 'TOB_R_PRICE', 'P2_compliance', 'Rev_other', 'Adult_daily_e-cig', 'Yth_daily_smokeless', 'R_Price_mp_estimate_ppp', 'P1_healthcare', 'NCD_CCS_TOB_SVY', 'TOBACCO_MPOWER_R_AFFORDABILITY', 'E_co

In [8]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'Date', 'IndicatorCode']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)

    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [9]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [10]:
retrieve_all()

100%|████████████████████████████████████████████████████████████████████████████████| 261/261 [14:52<00:00,  3.42s/it]


In [11]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [12]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [12]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 285/285 [00:00<00:00, 332.93it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [13]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 285/285 [00:00<00:00, 490.53it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [14]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 285/285 [00:02<00:00, 100.08it/s] 


Tổng số điểm dữ liệu là: 297817
Hoàn thành xử lý: alcohol_consumption


In [15]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|██████████| 285/285 [00:01<00:00, 234.59it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [13]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|███████████████████████████████████████████████████████████████████████████████| 546/546 [00:00<00:00, 878.82it/s]


Tổng số điểm dữ liệu là: 141804
Hoàn thành xử lý: cholesterol


In [17]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 285/285 [00:00<00:00, 12390.79it/s]

Tổng số điểm dữ liệu là: 21630
Hoàn thành xử lý: glucose


In [18]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 285/285 [00:00<00:00, 4383.04it/s]


Tổng số điểm dữ liệu là: 35523
Hoàn thành xử lý: physical_activities


In [19]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 285/285 [00:00<00:00, 913.45it/s]


Tổng số điểm dữ liệu là: 170916
Hoàn thành xử lý: infrastructure


In [14]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|████████████████████████████████████████████████████████████████████████████████| 546/546 [00:15<00:00, 34.32it/s]


Tổng số điểm dữ liệu là: 707018
Hoàn thành xử lý: tobacco


In [21]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 285/285 [00:00<00:00, 518.06it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
